In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import json
from scipy.optimize import minimize


# Load market configuration
with open("../../markets.json", "r") as f:
    config = json.load(f)

name = "primary_total_market"
BASE = Path("../../data")
print(BASE)
price_file = BASE / f"{name}_prices_monthly.csv"
returns_file = BASE / f"{name}_returns_monthly.csv"

INITIAL_INVESTMENT = config["initial_investment"]
MONTHLY_INVESTMENT = config["monthly_investment"]

../../data


In [2]:
LOOKBACK = 12          # number of months used to estimate return + covariance
RISK_AVERSION = 3.0    # higher value = more conservative
WEIGHT_CAP = 0.6       # max 60% in any single asset

In [3]:
rets = pd.read_csv(returns_file, index_col=0, parse_dates=True)
rets = rets.dropna()

In [4]:
print("Shape:", rets.shape)
print("Date range:", rets.index.min().date(), "to", rets.index.max().date())
print("Columns:", list(rets.columns))

Shape: (4712, 6)
Date range: 2007-04-11 to 2025-12-30
Columns: ['BND', 'GLD', 'TIP', 'TLT', 'VEU', 'VGT']


In [5]:
# Quick sanity checks
print("\nMissing values per column:\n", rets.isna().sum())
print("\nMonthly return stats:\n", rets.describe().loc[["mean","std","min","max"]])


Missing values per column:
 BND    0
GLD    0
TIP    0
TLT    0
VEU    0
VGT    0
dtype: int64

Monthly return stats:
            BND       GLD       TIP       TLT       VEU       VGT
mean  0.000127  0.000439  0.000144  0.000169  0.000274  0.000707
std   0.003297  0.011019  0.004005  0.009621  0.013816  0.014857
min  -0.054385 -0.087808 -0.029532 -0.066683 -0.115289 -0.134863
max   0.042201  0.112905  0.044537  0.075196  0.135958  0.134809


In [6]:
mu = rets.mean().values
cov = rets.cov().values
assets_list = rets.columns.tolist()

no_of_assets = len(assets_list)

mu, cov, assets_list


(array([0.00012691, 0.00043893, 0.000144  , 0.00016938, 0.00027361,
        0.0007074 ]),
 array([[ 1.08681918e-05,  8.12057808e-06,  9.19795598e-06,
          2.33640838e-05, -1.13849201e-07, -9.81262162e-07],
        [ 8.12057808e-06,  1.21410483e-04,  1.22914549e-05,
          1.83304732e-05,  2.56516646e-05,  6.88358823e-06],
        [ 9.19795598e-06,  1.22914549e-05,  1.60430548e-05,
          2.70059993e-05, -4.15980205e-06, -6.10199497e-06],
        [ 2.33640838e-05,  1.83304732e-05,  2.70059993e-05,
          9.25544732e-05, -4.10596802e-05, -3.83724807e-05],
        [-1.13849201e-07,  2.56516646e-05, -4.15980205e-06,
         -4.10596802e-05,  1.90868518e-04,  1.62169717e-04],
        [-9.81262162e-07,  6.88358823e-06, -6.10199497e-06,
         -3.83724807e-05,  1.62169717e-04,  2.20733784e-04]]),
 ['BND', 'GLD', 'TIP', 'TLT', 'VEU', 'VGT'])

In [7]:
def objective(weights):
    exp_ret = np.dot(weights, mu)
    var = weights.T @ cov @ weights
    return -(exp_ret + RISK_AVERSION * var)

In [9]:
constraints = [{"type": "eq", "fun": lambda w: np.sum(w) - 1}]  # weights must sum to 1. how this works is that the lambda function takes the weights as input and returns the sum of weights minus 1. The optimizer will try to find weights such that this function returns 0, which means the sum of weights is 1.
bounds = [(0, WEIGHT_CAP) for _ in range(no_of_assets)]  # the 0 is for no short selling, the WEIGHT_CAP is for max weight cap

bounds


[(0, 0.6), (0, 0.6), (0, 0.6), (0, 0.6), (0, 0.6), (0, 0.6)]

In [13]:
w0 = np.ones(no_of_assets) / no_of_assets  # start with equal weights
result = minimize(objective, w0, bounds=bounds, constraints=constraints)

if not result.success:
    raise ValueError("Optimization failed:", result.message)

In [15]:
weight_star = result.x

weights = pd.Series(weight_star, index=assets_list)

weights

BND    0.166667
GLD    0.166667
TIP    0.166667
TLT    0.166667
VEU    0.166667
VGT    0.166667
dtype: float64

In [16]:
print((weights * 100).round(2).astype(str) + "%")
print("\nSum of weights:", weights.sum().round(6))

BND    16.67%
GLD    16.67%
TIP    16.67%
TLT    16.67%
VEU    16.67%
VGT    16.67%
dtype: object

Sum of weights: 1.0


In [18]:
# show expected return + volatility (monthly)
port_ret = float(weight_star @ mu)
port_vol = float(np.sqrt(weight_star @ cov @ weight_star))
print("\nExpected monthly return:", round(port_ret, 5))
print("Expected monthly volatility:", round(port_vol, 5))


Expected monthly return: 0.00031
Expected monthly volatility: 0.00542


In [ ]:
def markowitz_weights(returns_window):
    """
    Given a rolling window of past returns,
    compute optimal portfolio weights using
    mean-variance optimization.

    Objective:
    Maximize: w'μ - λ w'Σw
    """

    # Expected returns vector (μ)
    mu = returns_window.mean().values

    # Covariance matrix (Σ)
    cov = returns_window.cov().values

    n = len(mu)  # number of assets

    # This function will be minimized.
    # We multiply by -1 because scipy minimizes by default.
    def objective(w):
        expected_return = w @ mu
        variance = w @ cov @ w
        return -(expected_return - RISK_AVERSION * variance)

    # Constraint: weights must sum to 1
    constraints = ({
        'type': 'eq',
        'fun': lambda w: np.sum(w) - 1
    })

    # Bounds: no shorting, no weight > cap
    bounds = tuple((0, WEIGHT_CAP) for _ in range(n))

    # Starting guess: equal weight
    w0 = np.ones(n) / n

    # Solve optimization problem
    result = minimize(objective,
                      w0,
                      bounds=bounds,
                      constraints=constraints)

    return result.x


In [ ]:
# m-o-m expected return = average return over past 12 months
average_return = rets.rolling(LOOKBACK).mean().shift(1)
average_return = average_return.dropna()

average_return

In [ ]:
def simulate():

    # Load monthly returns
    returns = pd.read_csv(returns_file, index_col=0, parse_dates=True)
    returns = returns.dropna()

    value = 10000   # current portfolio value
    values = []             # store portfolio value each month
    drawdowns = []          # store drawdown values
    weights_list = []       # store chosen weights

    peak = 10000    # track highest value seen so far

    # Loop through time, starting after lookback window
    for t in range(LOOKBACK, len(returns)):

        # 1. Use past LOOKBACK months to estimate μ and Σ
        window = returns.iloc[t-LOOKBACK:t]

        # 2. Compute optimal weights
        w = markowitz_weights(window)

        # 3. Apply weights to next month's returns
        r = returns.iloc[t].values
        portfolio_return = w @ r

        # 4. Update portfolio value
        value *= (1 + portfolio_return)

        # 5. Update drawdown
        peak = max(peak, value)
        dd = value / peak - 1

        values.append(value)
        drawdowns.append(dd)
        weights_list.append(w)

    dates = returns.index[LOOKBACK:]

    # Store performance results
    results = pd.DataFrame({
        "Value": values,
        "Drawdown": drawdowns
    }, index=dates)

    # Store time-varying weights
    weights_df = pd.DataFrame(
        weights_list,
        columns=returns.columns,
        index=dates
    )

    # Save outputs
    
    # Save outputs
    output_dir = BASE / "outputs"
    output_dir.mkdir(parents=True, exist_ok=True)

    results.to_csv(output_dir / "markowitz_results.csv")
    weights_df.to_csv(output_dir / "markowitz_weights.csv")

    print("Saved Markowitz outputs.")

    # Plot portfolio value
    plt.plot(results.index, results["Value"])
    plt.title("Markowitz Portfolio Value")
    plt.show()

    # Plot drawdown
    plt.plot(results.index, results["Drawdown"])
    plt.title("Markowitz Drawdown")
    plt.show()


In [ ]:
if __name__ == "__main__":
    simulate()